# 02 · Data conversion

Converts the raw datasets into a paired layout Ultralytics can read: same-named visible and infrared images at 640 px, YOLO labels and three data YAMLs per dataset. Every choice is recorded in `dataset_card.json`.

| | |
|---|---|
| **Kaggle settings** | Accelerator **GPU T4 x2**, Internet **on** |
| **Inputs to attach** | `cffm-net-pilot`, raw LLVIP and M3FD |
| **Expected runtime** | about 5 minutes |
| **Produces** | `data/llvip/`, `data/m3fd/` (and, if you save a version, an attachable output) |

In [ ]:
# Same setup cell in every notebook: find the project, install it, check the GPU.
import glob, shutil, subprocess, sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    PROJECT = Path("/kaggle/working/cffm-net-pilot")
    if not PROJECT.exists():
        # Kaggle mounts datasets at different depths under /kaggle/input, so search five levels.
        hits = [Path(p).parent for k in range(1, 6) for p in glob.glob("/kaggle/input" + "/*" * k + "/pyproject.toml")
                if (Path(p).parent / "src" / "cffm").is_dir()]
        # Earlier notebooks' outputs hold stale copies of the code, so prefer the uploaded dataset.
        hits.sort(key=lambda h: any((h.parent / d).exists() for d in ("runs", "data")))
        zips = [Path(p) for k in range(1, 5) for p in glob.glob("/kaggle/input" + "/*" * k + ".zip")
                if "cffm" in Path(p).name]
        if hits:
            shutil.copytree(hits[0], PROJECT, copy_function=shutil.copyfile)
        elif zips:                       # the zip was attached but not unpacked by Kaggle
            shutil.unpack_archive(str(zips[0]), "/kaggle/working")   # it holds one folder, cffm-net-pilot/
        assert PROJECT.exists(), "Add the 'cffm-net-pilot' dataset (the uploaded zip) as an input to this notebook."
        for p in [PROJECT, *PROJECT.rglob("*")]:     # inputs are read-only; our copy must be writable
            p.chmod(p.stat().st_mode | 0o200)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.171",
                    "faster-coco-eval>=1.6.7", "cloudpickle", "pytest"], check=True)   # cloudpickle: two-GPU launcher
    # Editable install, so DataLoader and DDP worker processes can import cffm too.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(PROJECT)], check=True)
else:
    PROJECT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "cffm").is_dir())
sys.path.insert(0, str(PROJECT / "src"))

import cffm
from cffm import env as cenv, pipeline
print("cffm", cffm.__version__, "imported from", Path(cffm.__file__).parent)   # must be inside PROJECT
env = cenv.setup()                       # also moves into the project folder
plan = pipeline.load_plan(env.project)   # configs/pilot.yaml

## Convert

Options come from `configs/pilot.yaml`, so every notebook builds the identical dataset.

In [ ]:
cards = {}
for name in ("llvip", "m3fd"):
    out = pipeline.prepare(name, env, plan, force=False)
    cards[name] = __import__("json").loads((out / "dataset_card.json").read_text())
    print(name, "->", out)

## What we made

The object-size bins show how small the objects are at training resolution.

In [ ]:
import pandas as pd
rows = [{"dataset": n, "train": c["images"]["train"], "val (full)": c["images"]["val"], "objects": c["objects"],
         **c["object_size_bins"]} for n, c in cards.items()]
pd.DataFrame(rows).set_index("dataset")

In [ ]:
print((env.data / "llvip" / "data_paired.yaml").read_text())

## Sanity checks

Every visible image has a same-size thermal partner and a label file, and every box lies inside the image.

In [ ]:
import cv2, numpy as np
for name in ("llvip", "m3fd"):
    root = env.data / name
    for split in ("train", "val"):
        vis = sorted((root / "images" / "visible" / split).glob("*.jpg"))
        missing = [v for v in vis if not (root / "images" / "infrared" / split / v.name).exists()
                   or not (root / "labels" / "visible" / split / (v.stem + ".txt")).exists()]
        boxes = np.array([list(map(float, l.split()[1:])) for f in (root / "labels" / "visible" / split).glob("*.txt")
                          for l in f.read_text().splitlines() if l.strip()], dtype=float).reshape(-1, 4)
        inside = ((boxes[:, :2] - boxes[:, 2:] / 2 >= -1e-6) & (boxes[:, :2] + boxes[:, 2:] / 2 <= 1 + 1e-6)).all()
        # every 25th pair is enough to catch a converter bug
        size_ok = all(cv2.imread(str(v)).shape[:2] == cv2.imread(str(root / "images" / "infrared" / split / v.name)).shape[:2]
                      for v in vis[::25])
        print(f"{name} {split}: {len(vis)} pairs, {len(missing)} incomplete, {len(boxes)} boxes, "
              f"all inside image: {bool(inside)}, pair sizes match: {size_ok}")
        assert vis and not missing and inside and size_ok

## Look at a few converted pairs, with their labels

A box beside the person in the thermal image means misregistration (notebook 03 measures it).

In [ ]:
import matplotlib.pyplot as plt
from cffm.data import read_pair
from cffm.viz import draw_pair

def show(name, k=3):
    root = env.data / name
    vis = sorted((root / "images" / "visible" / "val").glob("*.jpg"))
    vis = vis[:: max(1, len(vis) // k)][:k]                      # k pairs spread over the split
    names = cards[name]["names"]
    for v in vis:
        pair = read_pair(str(v)); h, w = pair.shape[:2]
        gt = []
        for line in (root / "labels" / "visible" / "val" / (v.stem + ".txt")).read_text().splitlines():
            c, x, y, bw, bh = map(float, line.split())
            gt.append([(x - bw / 2) * w, (y - bh / 2) * h, (x + bw / 2) * w, (y + bh / 2) * h, 1.0, c])
        plt.figure(figsize=(14, 4.5)); plt.imshow(draw_pair(pair, gt, names)); plt.axis("off"); plt.title(f"{name}: {v.name}")
        plt.show()

show("llvip"); show("m3fd")

### Keep the results

On Kaggle, click **Save Version → Save & Run All (Commit)** so `runs/` becomes an output that notebooks **09**
and **15** can attach (*Add Input → Your Work → this notebook*). Without a saved version the checkpoints vanish.